In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "464b9b4a",
   "metadata": {},
   "source": [
    "In the previous session we trained a model for predicting churn and evaluated it. Now let's deploy it"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "b3ff7c44",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.model_selection import KFold\n",
    "\n",
    "from sklearn.feature_extraction import DictVectorizer\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import roc_auc_score"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "81ee3ecc",
   "metadata": {},
   "outputs": [],
   "source": [
    "df = pd.read_csv('data-week-3.csv')\n",
    "\n",
    "df.columns = df.columns.str.lower().str.replace(' ', '_')\n",
    "\n",
    "categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)\n",
    "\n",
    "for c in categorical_columns:\n",
    "    df[c] = df[c].str.lower().str.replace(' ', '_')\n",
    "\n",
    "df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')\n",
    "df.totalcharges = df.totalcharges.fillna(0)\n",
    "\n",
    "df.churn = (df.churn == 'yes').astype(int)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "0b1903b3",
   "metadata": {},
   "outputs": [],
   "source": [
    "df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "ab4132a4",
   "metadata": {},
   "outputs": [],
   "source": [
    "numerical = ['tenure', 'monthlycharges', 'totalcharges']\n",
    "\n",
    "categorical = [\n",
    "    'gender',\n",
    "    'seniorcitizen',\n",
    "    'partner',\n",
    "    'dependents',\n",
    "    'phoneservice',\n",
    "    'multiplelines',\n",
    "    'internetservice',\n",
    "    'onlinesecurity',\n",
    "    'onlinebackup',\n",
    "    'deviceprotection',\n",
    "    'techsupport',\n",
    "    'streamingtv',\n",
    "    'streamingmovies',\n",
    "    'contract',\n",
    "    'paperlessbilling',\n",
    "    'paymentmethod',\n",
    "]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "92708443",
   "metadata": {},
   "outputs": [],
   "source": [
    "def train(df_train, y_train, C=1.0):\n",
    "    dicts = df_train[categorical + numerical].to_dict(orient='records')\n",
    "\n",
    "    dv = DictVectorizer(sparse=False)\n",
    "    X_train = dv.fit_transform(dicts)\n",
    "\n",
    "    model = LogisticRegression(C=C, max_iter=1000)\n",
    "    model.fit(X_train, y_train)\n",
    "    \n",
    "    return dv, model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "08ac302c",
   "metadata": {},
   "outputs": [],
   "source": [
    "def predict(df, dv, model):\n",
    "    dicts = df[categorical + numerical].to_dict(orient='records')\n",
    "\n",
    "    X = dv.transform(dicts)\n",
    "    y_pred = model.predict_proba(X)[:, 1]\n",
    "\n",
    "    return y_pred"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "2b01b2b1",
   "metadata": {},
   "outputs": [],
   "source": [
    "C = 1.0\n",
    "n_splits = 5"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "ce936aca",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "C=1.0 0.841 +- 0.008\n"
     ]
    }
   ],
   "source": [
    "kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)\n",
    "\n",
    "scores = []\n",
    "\n",
    "for train_idx, val_idx in kfold.split(df_full_train):\n",
    "    df_train = df_full_train.iloc[train_idx]\n",
    "    df_val = df_full_train.iloc[val_idx]\n",
    "\n",
    "    y_train = df_train.churn.values\n",
    "    y_val = df_val.churn.values\n",
    "\n",
    "    dv, model = train(df_train, y_train, C=C)\n",
    "    y_pred = predict(df_val, dv, model)\n",
    "\n",
    "    auc = roc_auc_score(y_val, y_pred)\n",
    "    scores.append(auc)\n",
    "\n",
    "print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "2f72b194",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "[0.8423083263338855,\n",
       " 0.8450681201165409,\n",
       " 0.8324061810154525,\n",
       " 0.8319390707936304,\n",
       " 0.8522598914373568]"
      ]
     },
     "execution_count": 17,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "scores"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "03e81326",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "0.8572386167896259"
      ]
     },
     "execution_count": 18,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)\n",
    "y_pred = predict(df_test, dv, model)\n",
    "\n",
    "y_test = df_test.churn.values\n",
    "auc = roc_auc_score(y_test, y_pred)\n",
    "auc"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3847f586",
   "metadata": {},
   "source": [
    "Save the model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "e3a452f1",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pickle"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "b4d16a27",
   "metadata": {},
   "outputs": [],
   "source": [
    "output_file = f'model_C={C}.bin'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "9533d28c",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "'model_C=1.0.bin'"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "output_file"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "f07f38d2",
   "metadata": {},
   "outputs": [],
   "source": [
    "f_out = open(output_file, 'wb') \n",
    "pickle.dump((dv, model), f_out)\n",
    "f_out.close()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "8887621f",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "-rwxrwxrwx 1 alexey alexey 2.5K Sep 30 14:10 'model_C=1.0.bin'\r\n"
     ]
    }
   ],
   "source": [
    "!ls -lh *.bin"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "8f85192f",
   "metadata": {},
   "outputs": [],
   "source": [
    "with open(output_file, 'wb') as f_out: \n",
    "    pickle.dump((dv, model), f_out)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "70d8a59e",
   "metadata": {},
   "source": [
    "Load the model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "8fe3b3eb",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pickle"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "007e9b49",
   "metadata": {},
   "outputs": [],
   "source": [
    "input_file = 'model_C=1.0.bin'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "e9dd11c3",
   "metadata": {},
   "outputs": [],
   "source": [
    "with open(input_file, 'rb') as f_in: \n",
    "    dv, model = pickle.load(f_in)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "5806ef9b",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "LogisticRegression(max_iter=1000)"
      ]
     },
     "execution_count": 8,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 27,
   "id": "16d96d50",
   "metadata": {},
   "outputs": [],
   "source": [
    "customer = {\n",
    "    'gender': 'female',\n",
    "    'seniorcitizen': 0,\n",
    "    'partner': 'yes',\n",
    "    'dependents': 'no',\n",
    "    'phoneservice': 'no',\n",
    "    'multiplelines': 'no_phone_service',\n",
    "    'internetservice': 'dsl',\n",
    "    'onlinesecurity': 'no',\n",
    "    'onlinebackup': 'yes',\n",
    "    'deviceprotection': 'no',\n",
    "    'techsupport': 'no',\n",
    "    'streamingtv': 'no',\n",
    "    'streamingmovies': 'no',\n",
    "    'contract': 'month-to-month',\n",
    "    'paperlessbilling': 'yes',\n",
    "    'paymentmethod': 'electronic_check',\n",
    "    'tenure': 1,\n",
    "    'monthlycharges': 29.85,\n",
    "    'totalcharges': 29.85\n",
    "}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 28,
   "id": "17ad51a5",
   "metadata": {},
   "outputs": [],
   "source": [
    "X = dv.transform([customer])"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 29,
   "id": "d0758291",
   "metadata": {},
   "outputs": [],
   "source": [
    "y_pred = model.predict_proba(X)[0, 1]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 30,
   "id": "79c7d951",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "input: {'gender': 'female', 'seniorcitizen': 0, 'partner': 'yes', 'dependents': 'no', 'phoneservice': 'no', 'multiplelines': 'no_phone_service', 'internetservice': 'dsl', 'onlinesecurity': 'no', 'onlinebackup': 'yes', 'deviceprotection': 'no', 'techsupport': 'no', 'streamingtv': 'no', 'streamingmovies': 'no', 'contract': 'month-to-month', 'paperlessbilling': 'yes', 'paymentmethod': 'electronic_check', 'tenure': 1, 'monthlycharges': 29.85, 'totalcharges': 29.85}\n",
      "output: 0.5912433520805763\n"
     ]
    }
   ],
   "source": [
    "print('input:', customer)\n",
    "print('output:', y_pred)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "163bfa7d",
   "metadata": {},
   "source": [
    "Making requests"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 40,
   "id": "934c7834",
   "metadata": {},
   "outputs": [],
   "source": [
    "import requests"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 41,
   "id": "68bf9932",
   "metadata": {},
   "outputs": [],
   "source": [
    "url = 'http://localhost:9696/predict'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 42,
   "id": "506e09a2",
   "metadata": {},
   "outputs": [],
   "source": [
    "customer = {\n",
    "    'gender': 'female',\n",
    "    'seniorcitizen': 0,\n",
    "    'partner': 'yes',\n",
    "    'dependents': 'no',\n",
    "    'phoneservice': 'no',\n",
    "    'multiplelines': 'no_phone_service',\n",
    "    'internetservice': 'dsl',\n",
    "    'onlinesecurity': 'no',\n",
    "    'onlinebackup': 'yes',\n",
    "    'deviceprotection': 'no',\n",
    "    'techsupport': 'no',\n",
    "    'streamingtv': 'no',\n",
    "    'streamingmovies': 'no',\n",
    "    'contract': 'two_year',\n",
    "    'paperlessbilling': 'yes',\n",
    "    'paymentmethod': 'electronic_check',\n",
    "    'tenure': 1,\n",
    "    'monthlycharges': 29.85,\n",
    "    'totalcharges': 29.85\n",
    "}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "72f4135e",
   "metadata": {},
   "outputs": [],
   "source": [
    "response = requests.post(url, json=customer).json()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 39,
   "id": "4b2c002a",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "{'churn': True, 'churn_probability': 0.5133820686195286}"
      ]
     },
     "execution_count": 39,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "response"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 27,
   "id": "8db38caf",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "sending email to asdx-123d\n"
     ]
    }
   ],
   "source": [
    "if response['churn']:\n",
    "    print('sending email to', 'asdx-123d')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a4a9a70b",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}